# Rompiendo el silencio estadístico: Violencia sexual contra niñas y adolescentes en Ecuador

**Investigador:** Pablo Andrés Japón Calva  
**Conferencia:** I Congreso Intersectorial sobre Violencia de Género y Mujeres en Situación de Vulnerabilidad (2025-12-05)  
**Datos:** Registro Estadístico de Egresos Hospitalarios (INEC 2019-2024), procesado en Google BigQuery.

---
## 1. Configuración del Entorno y Carga de Datos
Este notebook reproduce de forma interactiva y científica los modelos estadísticos inferenciales (Regresión Lineal Simple y contrastes de hipótesis Chi-Cuadrado) desarrollados para la ponencia.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.dpi"] = 150
print("Bibliotecas importadas exitosamente.")

## 2. Prevalencia de Diagnósticos de Maltrato (CIE-10 T74)
Analizamos los 2,067 ingresos hospitalarios registrados en Ecuador entre 2019 y 2024 bajo la categoría de síndromes de maltrato.

In [ ]:
df_prev = pd.read_csv("../data/processed/01_prevalencia_sindromes_maltrato_2019_2024.csv")
display(df_prev)

plt.figure(figsize=(9, 4.5))
palette = ["#C53030", "#4A5568", "#718096", "#A0AEC0", "#CBD5E0", "#E2E8F0"]
sns.barplot(data=df_prev, y="descripcion_diagnostico", x="total_casos_acumulados", palette=palette)
plt.title("Predominancia de Diagnósticos de Maltrato (CIE-10 T74) - Ecuador 2019-2024")
plt.xlabel("Total de Casos Acumulados")
plt.ylabel("")
plt.show()

## 3. Hipótesis 1: Regresión Lineal y Proyección 2025 (Abuso Sexual en Mujeres)
Modelamos la serie temporal anual post-pandemia (2020-2024) para contrastar la hipótesis de incremento lineal y proyectar el cierre de 2025.

In [ ]:
df_anual = pd.read_csv("../data/processed/02_casos_t742_anuales_por_sexo_2019_2024.csv")
display(df_anual)

# Filtrar período post-pandemia (2020-2024)
df_post = df_anual[df_anual["anio"] >= 2020].copy()
X = sm.add_constant(df_post["anio"])
y = df_post["mujeres"]

model = sm.OLS(y, X).fit()
print(model.summary())

pred_2025 = model.get_prediction([1, 2025]).summary_frame(alpha=0.05)
print(f"\n>>> PROYECCIÓN 2025: {pred_2025['mean'].values[0]:.1f} casos")
print(f">>> Intervalo 95% Confianza Media: [{pred_2025['mean_ci_lower'].values[0]:.1f}, {pred_2025['mean_ci_upper'].values[0]:.1f}]")

## 4. Hipótesis 2: Prueba Chi-Cuadrado de Bondad de Ajuste por Edad
Evaluamos si los casos de abuso sexual en mujeres se distribuyen uniformemente entre los 5 grupos etarios.

In [ ]:
df_edad = pd.read_csv("../data/processed/03_casos_t742_mujeres_por_grupo_edad.csv")
display(df_edad)

chi2, p_val = stats.chisquare(f_obs=df_edad["casos_observados"], f_exp=df_edad["frecuencia_esperada"])
print(f"Chi-cuadrado: {chi2:.3f}")
print(f"P-valor: {p_val:.4e} (p < 0.001: Rechazo contundente de H0)")

## 5. Hipótesis 3: Prueba Chi-Cuadrado de Independencia (Rural vs. Urbano)
Evaluamos si la proporción territorial rural/urbana ha variado significativamente entre 2019 y 2024.

In [ ]:
df_area = pd.read_csv("../data/processed/05_casos_t742_mujeres_rural_urbano_2019_2024.csv")
display(df_area)

contingencia = df_area[["area_rural", "area_urbana"]].values
chi2_ind, p_ind, dof, _ = stats.chi2_contingency(contingencia)
print(f"Chi-cuadrado de Independencia: {chi2_ind:.3f}")
print(f"Grados de libertad: {dof}")
print(f"P-valor: {p_ind:.3f} (p = 0.921 > 0.05: NO significativo, distribución estable)")